In [1]:
import pandas as pd
import numpy as np
import warnings
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, f1_score, roc_auc_score,
    classification_report, confusion_matrix
)

In [2]:
from google.colab import drive
drive.mount('/content/drive')

df = pd.read_csv('/content/drive/MyDrive/MINI_Project_BATCH8/Datasets/lung_donor_recipient_large_dataset.csv',low_memory=False)

Mounted at /content/drive


In [3]:
df = df.drop(columns=["Donor_ID", "Recipient_ID","Compatibility_Score"])


In [4]:
df["HLA_A_Match"]      = (df["Donor_HLA_A"]  == df["Recipient_HLA_A"]).astype(int)
df["HLA_B_Match"]      = (df["Donor_HLA_B"]  == df["Recipient_HLA_B"]).astype(int)
df["HLA_DR_Match"]     = (df["Donor_HLA_DR"] == df["Recipient_HLA_DR"]).astype(int)
df["HLA_Total_Match"]  = df["HLA_A_Match"] + df["HLA_B_Match"] + df["HLA_DR_Match"]
df["Blood_Type_Match"] = (df["Donor_Blood_Type"] == df["Recipient_Blood_Type"]).astype(int)
df["Age_Diff"]         = abs(df["Donor_Age"]           - df["Recipient_Age"])
df["Lung_Cap_Diff"]    = abs(df["Donor_Lung_Capacity"] - df["Recipient_Lung_Capacity"])


In [5]:
X = df.drop("Match_Label", axis=1)
y = df["Match_Label"]

In [6]:
categorical_cols = [
    "Donor_Gender", "Donor_Blood_Type", "Donor_HLA_A", "Donor_HLA_B", "Donor_HLA_DR",
    "Donor_Smoking_History", "Donor_Medical_History",
    "Recipient_Gender", "Recipient_Blood_Type", "Recipient_HLA_A", "Recipient_HLA_B",
    "Recipient_HLA_DR", "Recipient_Medical_History", "Recipient_Oxygen_Support",
    "Recipient_Urgency_Level"
]

numeric_cols = [
    "Donor_Age", "Donor_Lung_Capacity",
    "Recipient_Age", "Recipient_Lung_Capacity",
    "HLA_A_Match", "HLA_B_Match", "HLA_DR_Match", "HLA_Total_Match",
    "Blood_Type_Match", "Age_Diff", "Lung_Cap_Diff"
]

In [7]:
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler",  StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
])

In [8]:
preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_transformer, numeric_cols),
    ("cat", categorical_transformer, categorical_cols)
])

In [9]:
model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier",   LogisticRegression(
        C=0.1,
        max_iter=1000,
        random_state=42
    ))
])

In [10]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [11]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(model, X_train, y_train, cv=cv, scoring="accuracy", n_jobs=-1)

print(f"CV Accuracy : {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")


CV Accuracy : 0.8707 ± 0.0044


In [12]:
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

In [13]:
train_acc = model.score(X_train, y_train)
test_acc  = accuracy_score(y_test, y_pred)
f1        = f1_score(y_test, y_pred)
auc       = roc_auc_score(y_test, y_prob)

In [14]:
print(f"\nTrain Accuracy  : {train_acc:.4f}")
print(f"Test  Accuracy  : {test_acc:.4f}")
print(f" Overfit Gap    : {train_acc - test_acc:+.4f}  (< 0.05 = healthy)")
print(f"F1 Score        : {f1:.4f}")
print(f"ROC-AUC         : {auc:.4f}")

print("\Classification Report:\n", classification_report(y_test, y_pred))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))


Train Accuracy  : 0.8708
Test  Accuracy  : 0.8790
 Overfit Gap    : -0.0082  (< 0.05 = healthy)
F1 Score        : 0.8921
ROC-AUC         : 0.8721
\Classification Report:
               precision    recall  f1-score   support

           0       1.00      0.76      0.86       500
           1       0.81      1.00      0.89       500

    accuracy                           0.88      1000
   macro avg       0.90      0.88      0.88      1000
weighted avg       0.90      0.88      0.88      1000

Confusion Matrix:
 [[379 121]
 [  0 500]]


<>:7: SyntaxWarning: invalid escape sequence '\C'
<>:7: SyntaxWarning: invalid escape sequence '\C'
/tmp/ipykernel_10755/623506989.py:7: SyntaxWarning: invalid escape sequence '\C'
  print("\Classification Report:\n", classification_report(y_test, y_pred))


In [15]:
sample = X_test.iloc[[0]]
prediction = model.predict(sample)[0]
confidence = model.predict_proba(sample)[0][prediction]


In [16]:

print(f"\Sample Prediction : {'Matched' if prediction == 1 else 'Unmatched'}")
print(f"   Confidence        : {confidence:.2%}")

\Sample Prediction : Matched
   Confidence        : 81.59%


<>:1: SyntaxWarning: invalid escape sequence '\S'
<>:1: SyntaxWarning: invalid escape sequence '\S'
/tmp/ipykernel_10755/3664570554.py:1: SyntaxWarning: invalid escape sequence '\S'
  print(f"\Sample Prediction : {'Matched' if prediction == 1 else 'Unmatched'}")


In [17]:
import pickle
filename = 'lung_transplant.sav'
pickle.dump(model,open(filename,'wb'))
loaded_model = pickle.load(open('lung_transplant.sav','rb'))

In [18]:
from importlib.metadata import version
print(version('scikit-learn'))

1.6.1
